In [16]:
import os, json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv(override=True)
os.environ["HF"] = os.getenv("HF")
LLM = ChatGroq(model=os.getenv("GROQ_MODEL"), api_key=os.getenv("GROQ_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7765.83it/s]


In [17]:
# write the function to take the nesscery feature from the transaction.json
def format_transaction(record: dict, metadata: dict):
    metadata["id"] = record.get("id")
    metadata["status"] = record.get("status")
    metadata["type"] = record.get("type")
    metadata["amount"] = record.get("amount")
    metadata["name"] = record.get("name")
    return metadata


loader = JSONLoader(
    file_path="transactions.json",
    jq_schema=".[]",
    text_content=False,
    metadata_func=format_transaction,
)

docs = loader.load()

# rewrite the page content in human redable format
for doc in docs:
    data = json.loads(doc.page_content)
    direction = "Sent to" if data.get("type") == "debit" else "Recived from"
    doc.page_content=(
        f"Transaction ID: {data.get("id")} | "
        f"{direction} {data.get("name")}    ({data.get("upiId")}) | "
        f"Amount: {data.get("amount")} | "
        f"Status: {data.get("status")} | "
        f"Method: {data.get("paymentMethod")} | "
        f"Date: {data.get("date")} | "
        f"Note: {data.get("note")} | "
        f"Bank Ref: {data.get("bankRef")}"
    )

In [18]:
#vector stores
vectorstore  =FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":20, "fetch_k":30}
)

In [19]:
#Promt and chain
promt = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a transaction assistant for phonepe style payment app.
        Answer only from the context provided. Do not guess or hallucinate.
        Use ruppe symbol from amount. Keep answer short and simple")
        <context>
        {context}
        </context>"""),
        ("human", "{input}")
    ]
)

document_chain = create_stuff_documents_chain(LLM, promt)
retriever_chain = create_retrieval_chain(retriever, document_chain)

In [20]:
#invoke

response = retriever_chain.invoke({"input":"What is the total amount I sent to DMart overall"})
response["answer"]

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}